# Making smaller files 

In [3]:
import gc
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

YEAR = 2024
SFX = {2024: "_2024", 2026: "", 2025: "_2025"}[YEAR]
FINAL = "/global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/final_datasets"
merged_dir = Path(f"{FINAL}/lowCF_merged_all_datasets_allcols_MCD12Q1mask{SFX}.parquet")
output_file = Path(f"{FINAL}/lowCF_filtered_merged_pps_smap_no2_domsoil_15cols_MCD12Q1mask{SFX}.parquet")

KEEP = ["grid_lat", "grid_lon", "h_precip_mm", "datetime", "no2_mean", "no2_std", "tz_name", "datetime_utc",
        "datetime_local", "time_diff_hours", "pulse_start", "sm_surface", "surface_temp", "soil_temp_layer1",
        "WFPS_c", "dominant_texture", "state_name", "timezone"]

In [4]:
parts = sorted(merged_dir.glob("*.parquet")) if merged_dir.is_dir() else [merged_dir]
available = pq.read_schema(parts[0]).names
cols = [c for c in KEEP if c in available]
print("keeping:", cols, "| missing:", [c for c in KEEP if c not in available])

if output_file.exists():
    output_file.unlink()
writer, schema, n_rows = None, None, 0
for p in parts:
    chunk = pd.read_parquet(p, columns=cols, engine="pyarrow")
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        schema = table.schema
        writer = pq.ParquetWriter(output_file, schema)
    else:
        table = table.cast(schema)          # keep every band's types identical to the first
    writer.write_table(table)
    n_rows += len(chunk)
    print(f"{p.name}: {len(chunk):,} rows")
    del chunk, table; gc.collect()
writer.close()
print(f"\nSaved {n_rows:,} rows to: {output_file}")

keeping: ['grid_lat', 'grid_lon', 'h_precip_mm', 'datetime', 'no2_mean', 'no2_std', 'tz_name', 'datetime_utc', 'datetime_local', 'time_diff_hours', 'pulse_start', 'sm_surface', 'surface_temp', 'soil_temp_layer1', 'WFPS_c', 'dominant_texture', 'state_name', 'timezone'] | missing: []
band_36.0N.parquet: 356,184 rows
band_36.5N.parquet: 396,576 rows
band_37.0N.parquet: 668,304 rows
band_37.5N.parquet: 1,002,456 rows
band_38.0N.parquet: 1,215,432 rows
band_38.5N.parquet: 1,042,848 rows
band_39.0N.parquet: 1,663,416 rows
band_39.5N.parquet: 2,452,896 rows
band_40.0N.parquet: 2,684,232 rows
band_40.5N.parquet: 2,963,304 rows
band_41.0N.parquet: 2,820,096 rows
band_41.5N.parquet: 2,302,344 rows
band_42.0N.parquet: 1,942,488 rows
band_42.5N.parquet: 1,979,208 rows
band_43.0N.parquet: 1,971,864 rows
band_43.5N.parquet: 1,744,200 rows
band_44.0N.parquet: 1,461,456 rows
band_44.5N.parquet: 1,384,344 rows
band_45.0N.parquet: 1,086,912 rows
band_45.5N.parquet: 840,888 rows
band_46.0N.parquet: 954,7